# 纯torch代码实现默认参数下的Embedding全过程


In [1]:
import json

import torch
# 加载权重文件
from safetensors.torch import load_file
from torch import nn


In [2]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

texts = ["Hello Word, a test sentence"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 从前一步得到的结果
token_ids = [9707, 9322, 11, 264, 1273, 11652, 151643]

vocab_size = 151669
# 查询得知

hidden_size = 1024
padding_idx = 151643

In [3]:
# 加载配置文件
config_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/config.json"

with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)
config

FileNotFoundError: [Errno 2] No such file or directory: '/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/config.json'

# 2. Embedding lookup
根据 token id 在 embedding 矩阵中索引对应向量

In [ ]:
# nn.Embedding
embed_tokens = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=hidden_size,
    padding_idx=padding_idx,
)
embed_tokens

In [ ]:
token_ids = torch.tensor(token_ids, dtype=torch.long)
token_ids


# 模型权重加载
后面就需要用到模型权重了。这里模型权重处理

In [ ]:
model_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/model.safetensors"

# 读取文件
state_dict = load_file(model_file, device="cpu")  # 返回 dict: key -> torch.Tensor

# 查看有哪些 key
print(list(state_dict.keys()))

In [ ]:
# 大致看起来没有异常的键，但还是整理一下
# 处理键名（如果需要）
# 例如：移除 "model." 前缀
new_state_dict = {}
for key, value in state_dict.items():
    new_key = key
    # 根据你的模型结构调整
    # new_key = key.replace("model.", "")
    new_state_dict[new_key] = value
print(list(new_state_dict.keys()))

In [ ]:
# 提取 embedding 权重
embedding_weights = state_dict['embed_tokens.weight']
embedding_weights.shape

In [ ]:
# 直接加载没有问题
embed_tokens.weight.data.copy_(embedding_weights)

In [ ]:
# 使用 nn.Embedding 查表
inputs_embeds = embed_tokens(token_ids)  # shape: [7, 1024]
print(inputs_embeds.shape)
print(inputs_embeds[0])


# 位置信息（Positional Encoding）——告诉模型“顺序”

In [ ]:
from transformers import DynamicCache

# kv缓存用的
past_key_values = DynamicCache()
print(f"past_key_values: {past_key_values}")

# past_seen_tokens 表示已处理过的 token 数量，用于计算当前输入在序列中的绝对位置。
past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
print(f"past_seen_tokens: {past_seen_tokens}")

# cache_position 表示当前输入在 KV 缓存中的位置索引，用于增量生成。
# 首次前向：past_seen_tokens = 0，cache_position = [0, 1, 2, ...]
# 后续生成：past_seen_tokens = 已生成长度，cache_position = [已生成长度, 已生成长度+1, ...]
cache_position = torch.arange(
    past_seen_tokens, past_seen_tokens + inputs_embeds.shape[0], device=inputs_embeds.device
)
print(f"cache_position: {cache_position}")

# position_ids 是每个 token 的绝对位置索引，用于位置编码（如 RoPE）。
# 形状：[1, seq_len]
# 内容：[[0, 1, 2, ...]] 或 [[past_seen_tokens, past_seen_tokens+1, ...]]
# position_ids = cache_position.unsqueeze(-1)
# 这里不考虑batch，也就不unsqueeze了
position_ids = cache_position
print(f"position_ids: {position_ids}")

# 计算 attention_mask
# 到这里padding_size是None
attention_mask = torch.ones_like(token_ids)
print(f"attention_mask: {attention_mask}")

# 构建推理模型

In [ ]:
# 初始的隐藏层就是上面Embedding lookup的结果
hidden_states = inputs_embeds
print(hidden_states[0])

## 定义RoPE
旋转位置编码（Rotary Position Embedding, RoPE），这是一种将位置信息编码到 Transformer 模型中的方法。
与传统的绝对位置编码不同，RoPE 通过旋转操作将相对位置信息直接嵌入到 query 和 key 向量中。
论文地址：https://arxiv.org/abs/2104.09864

注意和transformer原始论文Attention is all you need中固定正弦/余弦位置编码（sinusoidal positional encoding, Sine-PE）区别和优点

In [ ]:
# 定义RoPE
from rotary_embedding import RotaryEmbedding
rotary_emb = RotaryEmbedding()
position_embeddings = rotary_emb(hidden_states, position_ids=position_ids)
print(f"position_embeddings: {position_embeddings[0][1]}")

## 开始逐层构建推理框架
### Layer层构成


### RMSNorm
RMSNorm（Root Mean Square Layer Normalization） 是一种归一化技术，相当于简化版的 LayerNorm。

他的前向计算步骤：
1. 计算输入的平方的均值（方差）
2. 用均方根的倒数来缩放输入
3. 乘以可学习的权重参数

在 Attention 中使用 RMSNorm 的好处
- 稳定注意力分数的计算：对 Q 和 K 进行归一化后，它们的点积（注意力分数）会更加稳定，防止数值过大或过小
- 提高训练稳定性：避免梯度爆炸或消失，特别是在深层模型中
- 计算效率更高：相比标准 LayerNorm，RMSNorm 不需要计算和减去均值，只需要计算均方根，运算更简单
- 适配大模型训练：在大规模语言模型（如 Qwen3）中，这种设计已被证明能提升性能和稳定性
- 改善注意力质量：归一化后的 Q 和 K 有助于注意力机制更好地捕捉相关性，而不受向量幅度的影响
- 这是现代 Transformer 架构（如 LLaMA、Qwen 等）的一个重要改进，相比原始的 Transformer 设计更加高效和稳定。



In [ ]:
from tests.learn_embedding.decode_layer import DecoderLayer
from tests.learn_embedding.rms_norm import RMSNorm
from tests.learn_embedding.rotary_embedding import RotaryEmbedding
# 根据模型配置构建transformer深度神经网络
layers = nn.ModuleList(
    [DecoderLayer(layer_idx) for layer_idx in range(config["num_hidden_layers"])]
)
norm = RMSNorm(
    config["hidden_size"],
    eps=config["rms_norm_eps"],
)
rotary_emb = RotaryEmbedding()

In [ ]:
print(layers)
print(norm)
print(rotary_emb)

In [ ]:
from embedding_model import EmbeddingModel
model = EmbeddingModel(
    embed_tokens=embed_tokens,
    layers=layers,
    norm=norm,
   rotary_emb=rotary_emb,
)
missing, unexpected = model.load_state_dict(state_dict,strict=True)
print("==> 加载完成")
if missing:
    print("missing keys:", missing[:20])
if unexpected:
    print("unexpected keys:", unexpected[:20])


# 推理！推理！推理！

In [ ]:
print(hidden_states)

In [ ]:
for layer in layers:
    hidden_states = layer(hidden_states, position_embeddings=position_embeddings)
    print(hidden_states)
hidden_states

In [ ]:
# 正规化
hidden_states = norm(hidden_states)
print(hidden_states)

In [ ]:
print(hidden_states)

那么，到这一步，transformer已经完成了推理，并且已经得到最终的输出结果。

接下来从sentence_transformer和vllm的源码分别处理后续过程，一般包括池化层和正则层

# sentence_transformer
## 池化层操作
Qwen3配置的是pooling_mode_lasttoken策略，也就是改进的最后一层的输出作为最终的输出。

In [ ]:
attention_mask

In [ ]:
seq_len, hidden_dim = hidden_states.shape  # 只有2个维度

# 找到最后一个有效token的位置
# attention_mask 例如: [1, 1, 1, 0, 0]
last_token_index = attention_mask.nonzero()[-1].item()  # 直接找最后一个1的索引
# 或者: last_token_index = (attention_mask == 1).nonzero()[-1].item()

# 提取该位置的embedding
embedding = hidden_states[last_token_index]  # 直接索引，形状 [hidden_dim]
print(embedding)

In [ ]:
# 正则化操作
embedding_result = F.normalize(embedding, p=2, dim=0)
print(embedding_result[:10].tolist())